Note: this nootbook if for preforming BERTTOpic clustering when the data is too much big to fit in memory leading to Oout of memory (OOM). The nootbook dideivde the data to chunks and  fit a sepreate berttopic model on eeach of the cunnkks. 

# Imports

In [ ]:
# ML
import pandas as pd
import numpy as np
import pyarrow  # ensures pandas can read/write parquet
import math

# Models
from bertopic import BERTopic
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import torch

# System
import gc
import json
from pathlib import Path
from datetime import datetime

# import subprocess
import sys
import fcntl
import tempfile
import os
import resource


# Pyton
from functools import wraps
from typing import Iterator, List, Tuple

# Visualization
import matplotlib.pyplot as plt
from IPython.display import display
from pprint import pprint
import matplotlib.ticker as ticker

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Parameters

In [ ]:
is_test = False

In [ ]:
input_data_path = None   # set by papermill or in next cell
min_topic_sizes = [2**8]
min_topic_size = None  # Legacy scalar parameter; overrides min_topic_sizes when provided.
chunk_size = 2**17
transform_batch_size = 25_000
time_windows = ["12H", "3D"]


In [ ]:
country = "Ecuador_Test"
if input_data_path is None:
    project_root = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
    input_data_path = str(
        project_root
        / "results"
        / "Labeled_Datasets"
        / country
        / f"{country}_part_all"
        / "step_1_translate_topics_entities"
        / f"{country}_part_all_step_1.gzip.parquet"
    )

required_columns = [
    "translated_post",
    "accountid",
    "post_time",
    "is_control",
]

if min_topic_size is not None:
    min_topic_sizes = [min_topic_size]

if is_test:
    print("Running in test mode. Only a small subset of the data will be processed.")
    min_topic_sizes = [10]
    chunk_size = 50

if not isinstance(min_topic_sizes, list) or not min_topic_sizes:
    raise ValueError("min_topic_sizes must be a non-empty list of positive integers.")
if any(
    isinstance(value, bool) or not isinstance(value, int) or value <= 0
    for value in min_topic_sizes
):
    raise ValueError("Every min_topic_sizes value must be a positive integer.")
if len(min_topic_sizes) != len(set(min_topic_sizes)):
    raise ValueError("min_topic_sizes must not contain duplicate values.")
if not isinstance(transform_batch_size, int) or transform_batch_size <= 0:
    raise ValueError("transform_batch_size must be a positive integer.")

print(f"Minimum topic sizes: {min_topic_sizes}")
print(f"Chunk size: {chunk_size:,}")
print(f"Transform batch size: {transform_batch_size:,}")
print(f"Input data path: {input_data_path}")


# Loading and Exporting


In [ ]:
from pathlib import Path


def get_dataset_part_path(path: str | Path) -> Path:
    """Return the dataset-part directory containing a pipeline step.

    Args:
        path: Input file or directory inside a ``step_*`` directory.

    Returns:
        Parent directory of the nearest ``step_*`` ancestor.

    Raises:
        ValueError: If the path is not inside a pipeline step directory.
    """
    input_path = Path(path).expanduser()
    for candidate in (input_path, *input_path.parents):
        if candidate.name.startswith("step_"):
            return candidate.parent
    raise ValueError("input_data_path must be inside a step_* directory")

In [ ]:
input_data_path = Path(input_data_path).expanduser()
dataset_part_path = get_dataset_part_path(input_data_path)
dataset_part = dataset_part_path.name
output_base_path = dataset_part_path / "step_2_BERTopic_clustering"
if is_test:
    output_base_path = output_base_path / "test"
output_base_path.mkdir(parents=True, exist_ok=True)
output_data_path = output_base_path / f"{dataset_part}_step_2.gzip.parquet"

print(f"Input data path: {input_data_path}")
print(f"Output base path: {output_base_path}")
print(f"Output data path: {output_data_path}")


In [ ]:
import pyarrow.parquet as pq

schema = pq.read_schema(input_data_path)

print(f"\nSchema of input data: {input_data_path}")
print("-" * 80)
print(f"{'Column':<40} {'Type'}")
print("-" * 80)

for field in schema:
    print(f"{field.name:<40} {field.type}")

print("-" * 80)
print(f"Total columns: {len(schema)}\n")

In [ ]:
print(f"Required columns: {required_columns}")

if is_test:
    parquet_file = pq.ParquetFile(input_data_path)
    batch = next(parquet_file.iter_batches(batch_size=100, columns=required_columns))
    df = batch.to_pandas()
else:
    df = pd.read_parquet(input_data_path, columns=required_columns, engine="pyarrow")

In [ ]:
print(f"df shape: {df.shape}")
print("df columns:")
pprint(list(df.columns))

In [ ]:
display(df.head())

# Utils

## Logs

In [ ]:
REPORT_PATH = output_base_path / "progress_report.txt"


def get_memory_usage_bytes() -> tuple[int, int]:
    """Return current and peak memory used by the SLURM task.

    Returns:
        Current and peak memory usage in bytes. Cgroup values include child
        processes; process-level values are used outside a cgroup.
    """
    try:
        cgroup_entry = next(
            line for line in Path("/proc/self/cgroup").read_text().splitlines()
            if line.startswith("0::")
        )
        relative_cgroup_path = cgroup_entry.split(":", maxsplit=2)[2]
        cgroup_path = (
            Path("/sys/fs/cgroup") / relative_cgroup_path.lstrip("/")
        )
        current_bytes = int((cgroup_path / "memory.current").read_text())
        peak_bytes = int((cgroup_path / "memory.peak").read_text())
        return current_bytes, peak_bytes
    except (OSError, StopIteration, ValueError):
        resident_pages = int(Path("/proc/self/statm").read_text().split()[1])
        current_bytes = resident_pages * os.sysconf("SC_PAGE_SIZE")
        peak_bytes = int(
            resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1_024
        )
        return current_bytes, peak_bytes


def format_memory(memory_bytes: int) -> str:
    """Format a byte count in gibibytes."""
    return f"{memory_bytes / 2**30:,.2f} GiB"


def write_report(msg: str) -> None:
    """Write a progress message with current and peak memory usage."""
    current_bytes, peak_bytes = get_memory_usage_bytes()
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = (
        f"{timestamp} | {msg} | memory_used={format_memory(current_bytes)} | "
        f"max_memory={format_memory(peak_bytes)}"
    )

    print(line)
    with open(REPORT_PATH, "a") as f:
        f.write(line + "\n")


write_report(f"Input data path: {input_data_path}")
write_report(f"Output base path: {output_base_path}")

def report_done(func):
    """Log whether a decorated function completed successfully."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            write_report(f"Completed: {func.__name__}")
            return result
        except Exception as e:
            write_report(f"FAILED: {func.__name__} | {e}")
            raise
    return wrapper


## Hardware Report

In [ ]:
import os
import platform
import subprocess
import threading
from pathlib import Path

import psutil
import torch

allocated_cpu_ids = sorted(os.sched_getaffinity(0))

cpu_info = subprocess.run(
    ["lscpu"],
    capture_output=True,
    text=True,
    check=True,
).stdout

cpu_model_name = "Unknown"

for line in cpu_info.splitlines():
    if line.startswith("Model name:"):
        cpu_model_name = line.split(":", 1)[1].strip()
        break

hardware_lines = [
    "Hardware and Slurm allocation",
    "-" * 75,
    f"Hostname:               {platform.node()}",
    f"CPU model:              {cpu_model_name}",
    f"Node physical cores:    {psutil.cpu_count(logical=False)}",
    f"Node logical CPUs:      {psutil.cpu_count(logical=True)}",
    f"Allocated CPUs:         {len(allocated_cpu_ids)}",
    f"Allocated CPU IDs:      {allocated_cpu_ids}",
    f"Slurm job ID:           {os.getenv('SLURM_JOB_ID', 'Not running under Slurm')}",
]

if torch.cuda.is_available():
    hardware_lines.extend([
        f"Allocated GPUs:         {torch.cuda.device_count()}",
        f"CUDA version:           {torch.version.cuda}",
    ])

    for gpu_index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(gpu_index)

        hardware_lines.extend([
            f"GPU {gpu_index} name:          {properties.name}",
            f"GPU {gpu_index} memory:        {properties.total_memory / 1024**3:.2f} GB",
            f"GPU {gpu_index} capability:    {properties.major}.{properties.minor}",
        ])
else:
    hardware_lines.append("Allocated GPUs:         0")

hardware_lines.append("-" * 75)
hardware_report = "\n".join(hardware_lines)
write_report(hardware_report)

In [ ]:
def start_gpu_keepalive(
    interval_seconds: float = 30.0,
) -> tuple[threading.Event | None, threading.Thread | None]:
    """Start a lightweight background CUDA heartbeat.

    Args:
        interval_seconds: Seconds between small GPU operations.

    Returns:
        Stop event and daemon thread, or two ``None`` values without CUDA.
    """
    if not torch.cuda.is_available():
        return None, None

    stop_event = threading.Event()

    def run_heartbeat() -> None:
        """Perform tiny matrix multiplications until asked to stop."""
        with torch.no_grad():
            heartbeat_matrix = torch.ones((32, 32), device="cuda")
            while not stop_event.is_set():
                torch.matmul(heartbeat_matrix, heartbeat_matrix)
                torch.cuda.synchronize()
                stop_event.wait(interval_seconds)

    keepalive_thread = threading.Thread(
        target=run_heartbeat,
        name="gpu-keepalive",
        daemon=True,
    )
    keepalive_thread.start()
    return stop_event, keepalive_thread


gpu_keepalive_stop, gpu_keepalive_thread = start_gpu_keepalive()
if gpu_keepalive_thread is not None:
    write_report("Started GPU keepalive with a 30-second interval.")

In [ ]:
write_report(" ############# running: 2_BERTopic_clustering.ipynb ############# ")
write_report(f"Today date: {datetime.now().strftime('%Y-%m-%d')}")
write_report(f"Input data path: {input_data_path}")
write_report(f"Output base path: {output_base_path}")
write_report(f"Loaded df: {input_data_path.name}.")
write_report(f"df shape: {df.shape}")
write_report(f"min_topic_sizes: {min_topic_sizes}")
print("df head:")
display(df.head(3))


## Save/Update df

In [ ]:
# @report_done
# def save_df(
#         df: pd.DataFrame,
#         output_data_path: Path = output_data_path
#     )-> None :
#     df.to_parquet(output_data_path, index=False, compression="gzip")
#     write_report(f"df is saved to: {output_data_path}")

In [ ]:
write_report(f"Loaded {len(df):,} rows from {input_data_path}")
df["accountid"] = df["accountid"].astype(str).map(sys.intern)
gc.collect()
write_report(f"Interned account IDs")

In [ ]:
from __future__ import annotations

import fcntl
import json
import os
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


@report_done
def save_df_columns_in_batches(
    df: pd.DataFrame,
    columns: list[str],
    output_path: str | Path,
    batch_size: int = 5_000,
    compression: str = "gzip",
) -> None:
    """Save selected DataFrame columns with bounded conversion memory.

    Args:
        df: Source DataFrame.
        columns: Columns to save in their current row order.
        output_path: Destination Parquet path.
        batch_size: Maximum rows converted to Arrow at once.
        compression: Parquet compression codec.
    """
    write_report(
        f"Starting save_df_columns_in_batches: rows={len(df):,}, "
        f"columns={columns}, batch_size={batch_size:,}, output_path={output_path}"
    )
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    writer: pq.ParquetWriter | None = None
    tmp_path: Path | None = None

    fd, tmp_name = tempfile.mkstemp(
        dir=output_path.parent,
        prefix=output_path.name + ".",
        suffix=".tmp",
    )
    os.close(fd)
    tmp_path = Path(tmp_name)

    try:
        for row_start in range(0, len(df), batch_size):
            row_end = min(row_start + batch_size, len(df))
            batch_df = df.iloc[row_start:row_end][columns]
            output_table = pa.Table.from_pandas(batch_df, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(
                    tmp_path,
                    output_table.schema,
                    compression=compression,
                )
            else:
                output_table = output_table.cast(writer.schema, safe=False)

            writer.write_table(output_table)
            del batch_df, output_table

        if writer is not None:
            writer.close()
            writer = None

        os.replace(tmp_path, output_path)
        tmp_path = None
        write_report(f"Saved staged columns to: {output_path}")
    finally:
        if writer is not None:
            writer.close()
        if tmp_path is not None:
            tmp_path.unlink(missing_ok=True)


@report_done
def update_df_columns_atomically(
    column_data_paths: list[str | Path],
    disk_df_path: str | Path,
    batch_size: int = 5_000,
    compression: str = "gzip",
) -> None:
    """Atomically merge staged Parquet columns into a shared Parquet file.

    All files are streamed in matching row batches, which keeps memory bounded
    regardless of the number of generated topic-size columns.

    Args:
        column_data_paths: Parquet files containing columns to add or update.
        disk_df_path: Existing shared Parquet file.
        batch_size: Maximum rows held in memory from each file at once.
        compression: Parquet compression codec.

    Raises:
        FileNotFoundError: If an input Parquet file does not exist.
        ValueError: If row counts differ or staged columns are duplicated.
        RuntimeError: If the atomic update fails.
    """
    write_report(
        f"Starting update_df_columns_atomically: "
        f"staged_files={len(column_data_paths):,}, batch_size={batch_size:,}, "
        f"disk_df_path={disk_df_path}"
    )
    disk_df_path = Path(disk_df_path)
    column_data_paths = [Path(path) for path in column_data_paths]

    all_paths = [disk_df_path, *column_data_paths]
    missing_paths = [path for path in all_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(f"Parquet file does not exist: {missing_paths[0]}")
    if not column_data_paths:
        raise ValueError("column_data_paths must contain at least one staged file.")

    lock_path = disk_df_path.with_suffix(disk_df_path.suffix + ".lock")
    tmp_path: Path | None = None
    writer: pq.ParquetWriter | None = None

    with open(lock_path, "w") as lock_file:
        fcntl.flock(lock_file, fcntl.LOCK_EX)

        try:
            disk_file = pq.ParquetFile(disk_df_path)
            disk_row_count = disk_file.metadata.num_rows
            column_files = [pq.ParquetFile(path) for path in column_data_paths]
            staged_columns = [
                column
                for column_file in column_files
                for column in column_file.schema.names
            ]

            if len(staged_columns) != len(set(staged_columns)):
                raise ValueError("Staged Parquet files contain duplicate columns.")
            for path, column_file in zip(column_data_paths, column_files):
                if column_file.metadata.num_rows != disk_row_count:
                    raise ValueError(
                        f"Row-count mismatch for {path}: "
                        f"{column_file.metadata.num_rows:,} rows instead of "
                        f"{disk_row_count:,}."
                    )

            write_report(
                f"[update_df_columns_atomically]\n"
                f"Updating columns: {staged_columns}\n"
                f"Disk path: {disk_df_path}"
            )

            fd, tmp_name = tempfile.mkstemp(
                dir=disk_df_path.parent,
                prefix=disk_df_path.name + ".",
                suffix=".tmp",
            )
            os.close(fd)
            tmp_path = Path(tmp_name)
            column_iterators = [
                column_file.iter_batches(batch_size=batch_size)
                for column_file in column_files
            ]
            row_count = 0

            for disk_batch in disk_file.iter_batches(batch_size=batch_size):
                batch_df = disk_batch.to_pandas()

                for column_iterator in column_iterators:
                    column_batch = next(column_iterator)
                    if len(column_batch) != len(disk_batch):
                        raise ValueError("Staged and shared Parquet batches are misaligned.")
                    column_df = column_batch.to_pandas()
                    for column in column_df.columns:
                        batch_df[column] = column_df[column].to_numpy(copy=False)
                    del column_df, column_batch

                output_table = pa.Table.from_pandas(batch_df, preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(
                        tmp_path,
                        output_table.schema,
                        compression=compression,
                    )
                else:
                    output_table = output_table.cast(writer.schema, safe=False)

                writer.write_table(output_table)
                row_count += len(disk_batch)
                del batch_df, output_table, disk_batch

            if row_count != disk_row_count:
                raise ValueError(
                    f"Processed {row_count:,} rows, expected {disk_row_count:,}."
                )

            if writer is not None:
                writer.close()
                writer = None

            os.replace(tmp_path, disk_df_path)
            tmp_path = None
            write_report(
                f"[update_df_columns_atomically]\nUpdated columns: "
                f"{', '.join(staged_columns)}\nSaved to: {disk_df_path}"
            )
        except Exception as error:
            raise RuntimeError(f"Failed to update Parquet file: {disk_df_path}") from error
        finally:
            if writer is not None:
                writer.close()
            if tmp_path is not None:
                tmp_path.unlink(missing_ok=True)
            fcntl.flock(lock_file, fcntl.LOCK_UN)


In [ ]:
# Initialize the shared file with every source column. The working DataFrame
# intentionally contains only the columns needed for topic modeling.

lock_path = output_data_path.with_suffix(output_data_path.suffix + ".lock")

with open(lock_path, "w") as lock_file:
    fcntl.flock(lock_file, fcntl.LOCK_EX)

    try:
        if not output_data_path.exists():
            shutil.copy2(input_data_path, output_data_path)
            write_report(f"Initialized shared output file: {output_data_path}")
        else:
            write_report(f"Shared output file already exists: {output_data_path}")
    finally:
        fcntl.flock(lock_file, fcntl.LOCK_UN)

# Posting Same Topic

In [ ]:
# time_windows is defined in the Parameters section.

In [ ]:
from collections import defaultdict, deque

@report_done
def add_posting_same_topic(
    df: pd.DataFrame,
    topic_column: str,
    time_window: str = "3D",
    account_id_column: str = "accountid",
    timestamp_column: str = "post_time",
) -> str:
    """Add prior authors posting the same topic within a time window.

    The DataFrame is modified in place. Author IDs are unique and ordered by
    the timestamps of their matching posts, from oldest to newest.

    Returns:
        Name of the added account-ID column.
    """
    write_report(
        f"Starting add_posting_same_topic: rows={len(df):,}, "
        f"topic_column={topic_column}, time_window={time_window}"
    )
    required_columns = {topic_column, account_id_column, timestamp_column}
    missing_columns = sorted(required_columns - set(df.columns))
    if missing_columns:
        raise KeyError(f"Missing required columns: {missing_columns}")

    output_column = f"posting_same_topic_{topic_column}_{time_window}_accountids"
    timestamps = pd.to_datetime(df[timestamp_column], errors="coerce").to_numpy(
        dtype="datetime64[ns]"
    )
    topics = df[topic_column].to_numpy()
    account_ids = df[account_id_column].to_numpy()
    window_delta = np.timedelta64(pd.Timedelta(time_window).value, "ns")

    # Store JSON strings directly to avoid retaining both Python lists and
    # their serialized copies for every row.
    output_account_ids = np.full(len(df), "[]", dtype=object)
    topic_history: dict[object, deque] = defaultdict(deque)

    for current_index in np.argsort(timestamps, kind="stable"):
        current_timestamp = timestamps[current_index]
        current_topic = topics[current_index]
        if np.isnat(current_timestamp) or pd.isna(current_topic):
            continue

        topic_posts = topic_history[current_topic]
        while topic_posts and current_timestamp - topic_posts[0][0] > window_delta:
            topic_posts.popleft()

        prior_account_ids = list(
            dict.fromkeys(account_ids[post_index] for _, post_index in topic_posts)
        )
        output_account_ids[current_index] = json.dumps(
            prior_account_ids,
            default=str,
        )
        topic_posts.append((current_timestamp, current_index))

    df[output_column] = output_account_ids

    write_report(
        f"[add_posting_same_topic] Added {output_column} for topic column "
        f"{topic_column} and time window {time_window}. "
        "Values were serialized as JSON strings for Parquet storage."
    )
    return output_column

In [ ]:
# Called after BERTopic creates the topic column.

# BERT English Topic Clustering

Note: the topic clustering doent influenced by the entities since it is preformed on the text without the entitites.

## Utils

### Posts Topic Stats

In [ ]:
@report_done
def make_topic_stats(df, topic_col_name):
    """
    Calculate per-topic post counts.

    Args:
        df (pd.DataFrame): Data with topics and 'is_control'.
        topic_col_name (str): Column name of topic labels.

    Returns:
        topic_stats (pd.DataFrame): Colums: [topic, total_count, io_Count, legitimate_posts, io_percent].
    """
    write_report(
        f"Starting make_topic_stats: rows={len(df):,}, "
        f"topic_column={topic_col_name}"
    )
    topic_stats = (
        df.groupby(topic_col_name)
          .agg(
              total_count=(topic_col_name, "count"),
              io_count=("is_control", lambda values: values.eq(False).sum())
          )
          .reset_index()
    )
    topic_stats["legitimate_posts"] = topic_stats["total_count"] - topic_stats["io_count"]
    topic_stats["io_percent"] = topic_stats["io_count"] / topic_stats["total_count"] * 100
    return topic_stats

In [ ]:
from matplotlib.lines import Line2D

def plot_topic_stats(topic_stats, topic_col_name, output_path: str | Path = ""):
    """
    Plot stacked IO vs legitimate post counts per topic.
    Saves the fig to output_path if provided.

    Args:
        topic_stats (pd.DataFrame): Output of make_topic_stats().
        topic_col_name (str): Topic column to plot.
        output_path (str): Path to *folder* save the plot.

    Returns:
        None
    """
    write_report(
        f"Starting plot_topic_stats: topics={len(topic_stats):,}, "
        f"topic_column={topic_col_name}, output_path={output_path}"
    )
    topics = topic_stats[topic_col_name]
    io_count = topic_stats["io_count"]
    legitimate_count = topic_stats["legitimate_posts"]
    io_percent = topic_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO posts
    bar_io = plt.bar(topics, io_count, label="IO posts", color="red")

    # 🔵 Top: Non-IO posts stacked above IO
    bar_legit = plt.bar(topics, legitimate_count, bottom=io_count, label="Non-IO posts")

    # Add IO percentage text
    ax = plt.gca()
    for x, io, legit, pct in zip(topics, io_count, legitimate_count, io_percent):
        total_height = io + legit
        ax.text(
            x,
            total_height + 0.5,              # slightly above the bar
            f"{pct:.1f}%",                 # format like 23.5%
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular"
        )

    plt.title(f"Post Count per Topic ({topic_col_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Posts")
    plt.xticks(topics)
    plt.grid(axis="y")
    plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

    percent_legend = Line2D([], [], color="none")
    total_posts_legend = Line2D([], [], color="none")
    
    plt.legend(
        handles=[bar_io, bar_legit, percent_legend, total_posts_legend],
        labels=[
            "IO posts",
            "Non-IO posts",
            "% IO Posts Ratio",
            f"Total posts: {topic_stats['total_count'].sum():,}",
        ],
        loc="best",
    )


    plt.tight_layout()


    if output_path:
        output_path = Path(output_path)
        plot_file_path = output_path / (f"{topic_col_name}_topic_clustering.png")
        plt.savefig(plot_file_path, dpi=300)
        print(f"plot is saved to {plot_file_path}")

    plt.show()

    plt.close()

### Accoutns Topic Stats

In [ ]:
@report_done
def make_topic_account_stats(
    df: pd.DataFrame,
    topic_col: str,
    author_col: str = "accountid",
    is_control_col: str = "is_control",
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Compute per-topic unique account counts (IO vs control) using an author-level threshold.

    Args:
        df: Post-level DataFrame.
        topic_col: Topic column name.
        author_col: Author identifier column.
        is_control_col: Column where 1 = control, 0 = IO.
        threshold: IO-rate threshold to label an author as IO (1 = IO).

    Returns:
        DataFrame with per-topic counts of unique IO/control accounts and IO percentage.
    """
    write_report(
        f"Starting make_topic_account_stats: rows={len(df):,}, "
        f"topic_column={topic_col}, threshold={threshold}"
    )
    # Author-level IO rate: 1 - mean(is_control)
    author_io_rate = 1 - df.groupby(author_col)[is_control_col].mean()
    author_io_tag = (author_io_rate >= threshold).astype(int)

    topic_author_df = df[[topic_col, author_col]].drop_duplicates().copy()
    topic_author_df["author_io_tag"] = topic_author_df[author_col].map(author_io_tag)

    stats = (
        topic_author_df
        .groupby(topic_col)["author_io_tag"]
        .value_counts()
        .unstack(fill_value=0)
    )

    # Ensure both columns exist even if absent in data
    stats = stats.reindex(columns=[0, 1], fill_value=0).rename(
        columns={0: "control_accounts", 1: "io_accounts"}
    ).reset_index()

    stats["total_accounts"] = stats["control_accounts"] + stats["io_accounts"]
    stats["io_percent"] = (stats["io_accounts"] / stats["total_accounts"]).mul(100)

    return stats

In [ ]:
def plot_topic_account_stats(
    topic_account_stats: pd.DataFrame,
    topic_col_name: str,
    output_path: Path | str | None = None,
):
    """Plot IO vs control account counts per topic.

    Args:
        topic_account_stats: Output of make_topic_account_stats().
        topic_col_name: Topic column name.
        output_path: Optional folder to save the plot.

    Returns:
        None
    """
    write_report(
        f"Starting plot_topic_account_stats: "
        f"topics={len(topic_account_stats):,}, topic_column={topic_col_name}, "
        f"output_path={output_path}"
    )
    topics = topic_account_stats[topic_col_name]
    io_count = topic_account_stats["io_accounts"]
    control_count = topic_account_stats["control_accounts"]
    io_percent = topic_account_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO accounts
    bar_io = plt.bar(topics, io_count, label="IO accounts", color="red")

    # 🔵 Top: Control accounts
    bar_control = plt.bar(
        topics,
        control_count,
        bottom=io_count,
        label="Control accounts",
    )

    # Add IO percentage text
    ax = plt.gca()
    for x, io, ctrl, pct in zip(topics, io_count, control_count, io_percent):
        total = io + ctrl
        ax.text(
            x,
            total + 0.3,
            f"{pct:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular",
        )

    plt.title(f"Unique Accounts per Topic ({topic_col_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Unique Accounts")
    plt.xticks(topics)
    plt.grid(axis="y")
    plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
    
    # Legend with extra info lines
    percent_legend = Line2D([], [], color="none")
    total_accounts_legend = Line2D([], [], color="none")

    plt.legend(
        handles=[bar_io, bar_control, percent_legend, total_accounts_legend],
        labels=[
            "IO accounts",
            "Control accounts",
            "% IO Accounts Ratio",
            f"Total accounts: {topic_account_stats['total_accounts'].sum():,}",
        ],
        loc="best",
    )

    if output_path:
        output_path = Path(output_path)
        plot_file_path = output_path / f"{topic_col_name}_topic_accounts.png"
        plt.savefig(plot_file_path, dpi=300)
        print(f"Plot saved to {plot_file_path}")

    plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
    plt.show()
    plt.close()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Embeddings

In [ ]:
import hashlib
import json


def fingerprint_texts(texts: list[str]) -> str:
    """Create a stable fingerprint for an ordered collection of texts.

    Args:
        texts: Texts in the same order used to generate embeddings.

    Returns:
        SHA-256 fingerprint of the ordered texts.
    """
    digest = hashlib.sha256()
    for text in texts:
        encoded_text = text.encode("utf-8")
        digest.update(len(encoded_text).to_bytes(8, byteorder="little"))
        digest.update(encoded_text)
    return digest.hexdigest()


def array_is_finite_in_batches(
    values: np.ndarray,
    batch_size: int = 100_000,
) -> bool:
    """Check numeric values without allocating a full-size boolean array.

    Args:
        values: Numeric array to validate.
        batch_size: Maximum rows checked at once.

    Returns:
        Whether every value is finite.
    """
    return all(
        np.isfinite(values[start : start + batch_size]).all()
        for start in range(0, len(values), batch_size)
    )


post_text_col = "translated_post"
embeddings_model_name = "digio/Twitter4SSE"
posts_texts = [str(text) for text in df[post_text_col].array]
texts_fingerprint = fingerprint_texts(posts_texts)

In [ ]:
embeddings_path = Path(input_data_path).parent / "post_embeddings.npy"
embeddings_metadata_path = (
    Path(input_data_path).parent / "post_embeddings_metadata.json"
)
embeddings_lock_path = Path(input_data_path).parent / "post_embeddings.lock"

with open(embeddings_lock_path, "w") as lock_file:
    fcntl.flock(lock_file, fcntl.LOCK_EX)
    can_use_existing_embeddings = False

    if embeddings_path.exists() and embeddings_metadata_path.exists():
        try:
            embeddings = np.load(embeddings_path)
            embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
            embeddings_metadata = json.loads(
                embeddings_metadata_path.read_text(encoding="utf-8")
            )
            can_use_existing_embeddings = (
                embeddings.ndim == 2
                and len(embeddings) == len(posts_texts)
                and array_is_finite_in_batches(embeddings)
                and embeddings_metadata.get("model_name") == embeddings_model_name
                and embeddings_metadata.get("text_column") == post_text_col
                and embeddings_metadata.get("row_count") == len(posts_texts)
                and embeddings_metadata.get("embedding_dimension") == embeddings.shape[1]
                and embeddings_metadata.get("texts_fingerprint") == texts_fingerprint
            )
        except (OSError, ValueError, KeyError, json.JSONDecodeError):
            can_use_existing_embeddings = False

    if can_use_existing_embeddings:
        write_report(
            f"Loaded validated embeddings from: {embeddings_path}, "
            f"shape: {embeddings.shape}"
        )
    else:
        from sentence_transformers import SentenceTransformer

        write_report("Generating missing, stale, or invalid embeddings.")
        model = SentenceTransformer(embeddings_model_name, device=device)
        embeddings = model.encode(
            posts_texts,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        embeddings_metadata = {
            "model_name": embeddings_model_name,
            "text_column": post_text_col,
            "row_count": len(posts_texts),
            "embedding_dimension": embeddings.shape[1],
            "texts_fingerprint": texts_fingerprint,
        }

        embedding_fd, embedding_tmp = tempfile.mkstemp(
            dir=embeddings_path.parent, prefix=embeddings_path.name, suffix=".tmp"
        )
        metadata_fd, metadata_tmp = tempfile.mkstemp(
            dir=embeddings_path.parent,
            prefix=embeddings_metadata_path.name,
            suffix=".tmp",
        )
        os.close(embedding_fd)
        os.close(metadata_fd)
        embedding_tmp_path = Path(embedding_tmp)
        metadata_tmp_path = Path(metadata_tmp)

        try:
            with embedding_tmp_path.open("wb") as embedding_file:
                np.save(embedding_file, embeddings)
            metadata_tmp_path.write_text(
                json.dumps(embeddings_metadata, indent=2), encoding="utf-8"
            )
            os.replace(embedding_tmp_path, embeddings_path)
            os.replace(metadata_tmp_path, embeddings_metadata_path)
        finally:
            embedding_tmp_path.unlink(missing_ok=True)
            metadata_tmp_path.unlink(missing_ok=True)

        write_report(
            f"Generated embeddings from column: {post_text_col}, "
            f"using model: {embeddings_model_name}, shape: {embeddings.shape}"
        )
        write_report(f"Saved embeddings to: {embeddings_path}")
        write_report(f"Saved embeddings metadata to: {embeddings_metadata_path}")

## BERTopic

[BERTopic Parametes](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#calculate_probabilities)

### Build Chunks Models

https://maartengr.github.io/BERTopic/getting_started/merge/merge.html

In [ ]:
from pathlib import Path
from typing import Iterator, List, Tuple

import gc

import numpy as np
import torch
from bertopic import BERTopic


def iter_text_chunks(
    texts: List[str],
    chunk_size: int,
) -> Iterator[Tuple[int, List[str]]]:
    """Yield fixed-size text chunks.

    Args:
        texts: Documents to split.
        chunk_size: Maximum documents per chunk.

    Yields:
        Pairs containing the chunk start index and documents.
    """
    for start in range(0, len(texts), chunk_size):
        yield start, texts[start : start + chunk_size]


@report_done
def build_and_save_chunk_models(
    texts: List[str],
    embeddings: np.ndarray,
    chunk_size: int,
    min_topic_size: int,
    output_dir: str | Path,
) -> None:
    """Train BERTopic models on chunks and save each model.

    Args:
        texts: Documents.
        embeddings: Precomputed embeddings with shape ``(n_docs, dim)``.
        chunk_size: Maximum documents per chunk.
        min_topic_size: Minimum HDBSCAN cluster size.
        output_dir: Directory in which to save chunk models.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    write_report(f"[build_and_save_chunk_models] Using device {device}")

    if device == "cuda":
        from cuml.cluster import HDBSCAN
        from cuml.manifold import UMAP

    write_report(
        f"Starting build_and_save_chunk_models: documents={len(texts):,}, "
        f"chunk_size={chunk_size:,}, min_topic_size={min_topic_size:,}, "
        f"device={device}, output_dir={output_dir}"
    )

    for idx, (start, text_chunk) in enumerate(
        iter_text_chunks(texts, chunk_size)
    ):
        end = start + len(text_chunk)
        emb_chunk = embeddings[start:end]

        if len(text_chunk) < min_topic_size:
            write_report(
                f"Skipping chunk {idx}, rows {start}:{end}: "
                f"{len(text_chunk)} documents is smaller than "
                f"min_topic_size={min_topic_size}."
            )
            continue

        if device == "cuda":
            torch.cuda.empty_cache()

            umap_model = UMAP(random_state=42,)

            hdbscan_model = HDBSCAN(
                min_cluster_size=min_topic_size,
            )

            model = BERTopic(
                language="english",
                umap_model=umap_model,
                hdbscan_model=hdbscan_model,
                low_memory=True,
                calculate_probabilities=False,
                min_topic_size=min_topic_size,
            )
        else:
            model = BERTopic(
                language="english",
                min_topic_size=min_topic_size,
                low_memory=True,
                calculate_probabilities=False,
            )

        model.fit(text_chunk, emb_chunk)

        model_path = output_dir / f"chunk_model_{idx}"
        model.save(model_path, serialization="safetensors")

        write_report(
            f"Saved chunk model {idx} for rows "
            f"{start}:{end} at {model_path}"
        )

        del model, emb_chunk, text_chunk
        gc.collect()

        if device == "cuda":
            torch.cuda.empty_cache()

### Merge Chunks Models

In [ ]:
@report_done
def load_and_merge_BERTopic_models(
    chunk_models_dir: str | Path,
    min_similarity: float = 0.7, # default value is 0.7 in BERTopic docs
) -> BERTopic:
  
    chunk_models_dir = Path(chunk_models_dir)
    paths = sorted(chunk_models_dir.iterdir())
    write_report(
        f"Starting load_and_merge_BERTopic_models: models={len(paths):,}, "
        f"min_similarity={min_similarity}, directory={chunk_models_dir}"
    )
    
    if not paths:
        raise ValueError(f"No chunk models found in {chunk_models_dir}")
    
    merged_model = None

    for path in paths:
        chunk_model = BERTopic.load(path)

        if merged_model is None:
            merged_model = BERTopic.merge_models(
                [chunk_model], min_similarity=min_similarity
            )
        else:
            merged_model = BERTopic.merge_models(
                [merged_model, chunk_model], min_similarity=min_similarity
            )

        del chunk_model
        gc.collect()


    if merged_model is None:
        raise RuntimeError("merged_model is None after loading chunk models.")
    

    # save merged model
    merge_model_file_name = f"merged_BERTopic_model_min_similarity_{min_similarity}"
    merged_model.save(
        chunk_models_dir / merge_model_file_name, serialization="safetensors"
    )
    write_report(f"Merged model saved to {chunk_models_dir / merge_model_file_name}")

    return merged_model

### Run BERTopic Modeling

In [ ]:
def transform_in_batches(
    model: BERTopic,
    texts: list[str],
    embeddings: np.ndarray,
    batch_size: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Assign BERTopic topics without building one large similarity matrix.

    Args:
        model: Fitted BERTopic model used for topic assignment.
        texts: Documents in the same order as the embeddings.
        embeddings: Document embeddings with one row per document.
        batch_size: Maximum documents transformed at once.

    Returns:
        Topic assignments and their corresponding probabilities.

    Raises:
        ValueError: If input lengths differ, the batch size is invalid, or the
            model returns full probability distributions instead of one value
            per document.
    """
    if len(texts) != len(embeddings):
        raise ValueError("texts and embeddings must have the same length.")
    if batch_size <= 0:
        raise ValueError("batch_size must be a positive integer.")

    topics = np.empty(len(texts), dtype=np.int32)
    probabilities = np.empty(len(texts), dtype=np.float32)
    total_batches = (len(texts) + batch_size - 1) // batch_size
    write_report(
        f"Starting transform_in_batches: documents={len(texts):,}, "
        f"batches={total_batches:,}, batch_size={batch_size:,}"
    )

    for batch_number, start in enumerate(
        range(0, len(texts), batch_size), start=1
    ):
        end = min(start + batch_size, len(texts))
        batch_topics, batch_probabilities = model.transform(
            texts[start:end], embeddings[start:end]
        )
        batch_probabilities = np.asarray(batch_probabilities)
        if batch_probabilities.ndim != 1:
            raise ValueError(
                "Batched transformation requires one probability per document; "
                "set calculate_probabilities=False on the BERTopic model."
            )

        topics[start:end] = np.asarray(batch_topics, dtype=np.int32)
        probabilities[start:end] = batch_probabilities.astype(
            np.float32, copy=False
        )
        if batch_number % 25 == 0 or batch_number == total_batches:
            write_report(
                f"Transformed batch {batch_number:,}/{total_batches:,} "
                f"through row {end:,}."
            )

    return topics, probabilities


def run_bertopic_for_min_topic_size(
    min_topic_size: int,
) -> tuple[list[str], Path]:
    """Run and save the complete BERTopic pipeline for one topic size.

    Args:
        min_topic_size: Minimum HDBSCAN cluster size for this run.

    Returns:
        Generated column names and their staged Parquet path.
    """
    global REPORT_PATH

    topic_col = f"BERTopic_topic_{min_topic_size}"
    probability_col = f"BERTopic_prob_{min_topic_size}"
    topic_top_words_col = f"{topic_col}_top_words"
    output_path = output_base_path / f"topic_col_{topic_col}"
    output_path.mkdir(parents=True, exist_ok=True)
    REPORT_PATH = output_path / "progress_report.txt"

    hardware_report_path = output_path / "hardware_info.txt"
    hardware_report_path.write_text(hardware_report, encoding="utf-8")
    write_report(
        f"Starting run_bertopic_for_min_topic_size: "
        f"min_topic_size={min_topic_size:,}, output_path={output_path}"
    )
    write_report(f"Output path: {output_path}")
    write_report(f"Wrote hardware report to: {hardware_report_path}")

    chunk_models_base_path = (
        output_path / f"BERTopic_models_min_topic_size_{min_topic_size}"
    )
    chunk_models_base_path.mkdir(parents=True, exist_ok=True)
    chunk_models_output_path = Path(
        tempfile.mkdtemp(prefix="run_", dir=chunk_models_base_path)
    )
    write_report(f"Chunk models will be saved to: {chunk_models_output_path}")

    build_and_save_chunk_models(
        texts=posts_texts,
        embeddings=embeddings,
        chunk_size=chunk_size,
        min_topic_size=min_topic_size,
        output_dir=chunk_models_output_path,
    )
    merged_model = load_and_merge_BERTopic_models(
        chunk_models_dir=chunk_models_output_path
    )
    topics, probabilities = transform_in_batches(
        model=merged_model,
        texts=posts_texts,
        embeddings=embeddings,
        batch_size=transform_batch_size,
    )

    df[topic_col] = topics
    df[probability_col] = probabilities
    topic_words_by_id = {}
    for topic_id in pd.unique(df[topic_col]):
        topic_words = merged_model.get_topic(topic_id)
        words = tuple(word for word, _ in topic_words) if topic_words else ()
        topic_words_by_id[topic_id] = json.dumps(words)
    df[topic_top_words_col] = df[topic_col].map(topic_words_by_id)
    del topics, probabilities, topic_words_by_id

    topic_columns = [topic_col, probability_col, topic_top_words_col]
    write_report(
        f"Prepared BERTopic columns for min_topic_size={min_topic_size}."
    )

    posting_columns = []
    ordered_time_windows = sorted(time_windows, key=pd.Timedelta, reverse=True)
    for time_window in ordered_time_windows:
        column = add_posting_same_topic(
            df=df,
            topic_column=topic_col,
            time_window=time_window,
        )
        posting_columns.append(column)

        assert len(df[column]) == len(df)
        non_empty_count = sum(value != "[]" for value in df[column].array)
        print(f"{column}: {non_empty_count:,} posts with prior same-topic authors")

        write_report(f"Prepared posting-same-topic column: {column}")

    try:
        fig = merged_model.visualize_topics(
            top_n_topics=100,
            custom_labels=True,
        )
        figure_path = output_path / f"{topic_col}_intertopic_distance.html"
        fig.write_html(figure_path)
        print(f"Topic visualization saved to: {figure_path}")

        slurm_job_id = os.environ.get("SLURM_JOB_ID")
        slurm_job_name = os.environ.get("SLURM_JOB_NAME", "")
        if slurm_job_id is None or slurm_job_name == "sinteractive":
            display(fig)
        del fig
    except Exception as error:
        print("Could not create the topic visualization.")
        print("Error:", error)

    display(merged_model.get_topic_info())
    topic_stats = make_topic_stats(df, topic_col)
    plot_topic_stats(topic_stats, topic_col, output_path)
    topic_account_stats = make_topic_account_stats(df, topic_col)
    display(topic_account_stats)
    plot_topic_account_stats(topic_account_stats, topic_col, output_path)

    generated_columns = topic_columns + posting_columns
    missing_columns = set(generated_columns) - set(df.columns)
    assert not missing_columns, (
        f"Missing prepared columns for min_topic_size={min_topic_size}: "
        f"{sorted(missing_columns)}"
    )
    del merged_model, topic_stats, topic_account_stats
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    staged_columns_path = output_path / "new_columns_df.parquet"
    save_df_columns_in_batches(
        df=df,
        columns=generated_columns,
        output_path=staged_columns_path,
    )
    write_report(f"Completed min_topic_size={min_topic_size}")

    df.drop(columns=generated_columns, inplace=True)
    gc.collect()

    return generated_columns, staged_columns_path

In [ ]:
all_generated_columns = []
staged_column_paths = []
try:
    for current_min_topic_size in min_topic_sizes:
        generated_columns, staged_columns_path = run_bertopic_for_min_topic_size(
            current_min_topic_size
        )
        all_generated_columns.extend(generated_columns)
        staged_column_paths.append(staged_columns_path)

    del embeddings, posts_texts
    df.drop(columns=[post_text_col], inplace=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    REPORT_PATH = output_base_path / "progress_report.txt"
    update_df_columns_atomically(
        staged_column_paths,
        output_data_path,
        batch_size=5_000,
    )
    write_report(
        "Saved all BERTopic runs to the shared output after every topic size completed."
    )
finally:
    if gpu_keepalive_stop is not None:
        gpu_keepalive_stop.set()
        gpu_keepalive_thread.join(timeout=5.0)
        write_report("Stopped GPU keepalive.")



In [ ]:
# Topic columns are created inside run_bertopic_for_min_topic_size.


In [ ]:
# Per-size results are saved with Parquet-compatible dtypes.


### Update and Save df to Disk



In [ ]:
# Per-size backup DataFrames are saved inside the run loop.


In [ ]:
# Topic columns are atomically added to the shared Parquet inside the run loop.


## Posting Same Topic

In [ ]:
# Posting-same-topic columns are created and saved for every topic size.


In [ ]:
# Historical single-size posting-same-topic example removed; the loop handles it.


## Validation

In [ ]:
# Per-size generated columns are validated immediately after each run.


### Graph and Statstistics

In [ ]:
# Intertopic-distance visualizations are generated inside the run loop.


In [ ]:
# Topic information is displayed and each merged model is released in the loop.


Topic Post Statstistics

In [ ]:
# Post-level topic statistics and plots are generated inside the run loop.


Topic Account Statstistics

In [ ]:
# Account-level topic statistics and plots are generated inside the run loop.


## Validate Data for the Next Step

In [ ]:
# The shared output must contain every source column and every generated column.
input_schema = pq.read_schema(input_data_path)
input_index_columns = {
    column
    for column in (input_schema.pandas_metadata or {}).get("index_columns", [])
    if isinstance(column, str)
}
input_data_columns = set(input_schema.names) - input_index_columns
final_columns = set(pq.read_schema(output_data_path).names)
expected_columns = input_data_columns | set(all_generated_columns)
missing_columns = expected_columns - final_columns
assert not missing_columns, f"Missing output columns: {sorted(missing_columns)}"

REPORT_PATH = output_base_path / "progress_report.txt"
write_report(f"Completed min_topic_sizes: {min_topic_sizes}")
write_report(f"Final output is complete: {output_data_path}")


In [ ]:
write_report("Finished Running Successfully")
write_report("#######################################################")
write_report("#######################################################")